In [1]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect("../data/processed/market_data.db")

fx_df  = pd.read_sql("SELECT * FROM fx_daily", conn, index_col="date", parse_dates=["date"])
eq_df  = pd.read_sql("SELECT * FROM equities_daily", conn, index_col="date", parse_dates=["date"])
rat_df = pd.read_sql("SELECT * FROM rates_daily", conn, index_col="date", parse_dates=["date"])

fx_ret  = fx_df.pct_change().dropna()
eq_ret  = eq_df.pct_change().dropna()

print("✅ Données chargées")


✅ Données chargées


In [4]:
import openpyxl
from openpyxl import Workbook

# Stats pour le VaR model
assets = {
    "EURUSD": fx_ret["eurusd"],
    "USDJPY": fx_ret["usdjpy"],
    "SPY":    eq_ret["SPY"],
    "XLK":    eq_ret["XLK"],
    "XLF":    eq_ret["XLF"],
}

rows = []
for name, ret in assets.items():
    rows.append({
        "Asset":           name,
        "Mean Daily Ret":  round(ret.mean(), 6),
        "Std Daily":       round(ret.std(), 6),
        "VaR 95% (1D)":    round(ret.quantile(0.05) * 100, 4),
        "VaR 99% (1D)":    round(ret.quantile(0.01) * 100, 4),
        "Ann. Vol %":      round(ret.std() * np.sqrt(252) * 100, 2),
        "Sharpe":          round((ret.mean() / ret.std()) * np.sqrt(252), 2),
        "Max Drawdown %":  round((((1+ret).cumprod() / (1+ret).cumprod().cummax()) - 1).min() * 100, 2),
    })

var_df = pd.DataFrame(rows)

# Export Excel
with pd.ExcelWriter("../models/excel/risk_model.xlsx", engine="openpyxl") as writer:
    var_df.to_excel(writer, sheet_name="VaR Model", index=False)
    fx_ret[["eurusd","usdjpy"]].to_excel(writer, sheet_name="FX Returns")
    rat_df.to_excel(writer, sheet_name="Rates Data")

print("✅ risk_model.xlsx créé dans models/excel/")


✅ risk_model.xlsx créé dans models/excel/


## 2. FX Analysis

### EUR/USD
- Trend: Bearish 2021-2022 driven by USD strength (Fed/ECB policy divergence)
- EUR/USD hit lows of ~1.035 in Sept 2022 as Fed hiked aggressively
- Partial recovery in 2023-2024 as ECB caught up with rate hikes
- **Realized Vol:** 7.06% annualized — structurally low vs equities
- **Sharpe Ratio:** -0.89 → euro underperformed risk-free rate over period
- **Max Drawdown:** -22.24%

### USD/JPY
- BoJ maintained ultra-loose policy while Fed hiked → massive carry trade
- USD/JPY rallied from 102 to 152 (2021-2022) — 50% move
- **Realized Vol:** 8.80% annualized
- **Sharpe Ratio:** 0.14 → modest positive return
- **Max Drawdown:** -14.75%

### Key Signal — IV vs RV Gap
- VIX (implied vol proxy) averaged 18-20 during 2023-2024
- EUR/USD realized vol stayed below 7%
- **Gap = ~11-13 vol points → vol structurally overpriced on EUR/USD**
- Trading implication: selling EUR/USD straddles or strangles was systematically profitable
